# 2 - Synthetic 2D Data Generation (Added Complexity through Ice-Cream)

The 2D synthetic system adds two things that are useful for MLTSA:

- more geometric complexity than the 1D toy example
- a **time-dependent change in importance**, because the discriminative direction grows
  as the trajectory evolves

The silly mental image is an ice-cream: a simple cone-like direction that matters for the
class, plus a spiral nuisance motion that makes the path look richer than it really is.

In [ ]:
from pathlib import Path
import sys

try:
    import mltsa  # noqa: F401
except ImportError:
    for parent in (Path.cwd(), *Path.cwd().parents):
        src_dir = parent / "src"
        if (src_dir / "mltsa").exists():
            sys.path.insert(0, str(src_dir))
            break
    import mltsa  # noqa: F401

import numpy as np
import matplotlib.pyplot as plt

for parent in (Path.cwd(), *Path.cwd().parents):
    notebooks_dir = parent / "notebooks"
    if notebooks_dir.exists():
        DATA_DIR = notebooks_dir / "_generated"
        break
else:
    DATA_DIR = Path.cwd() / "_generated"

DATA_DIR.mkdir(exist_ok=True, parents=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.set_printoptions(precision=3, suppress=True)

## Step 1: serve the ice-cream

First we create the actual dataset. In this 2D toy system, the `x` direction is the
informative one, while the `y` direction acts as the nuisance spiral.

In [ ]:
from mltsa.synthetic import make_2d_dataset

dataset = make_2d_dataset(
    n_trajectories=160,  # Number of trajectories to generate
    n_steps=80,  # Number of time steps per trajectory
    n_features=16,  # Number of observed projected features
    pattern="spiral",  # The nuisance path we want to use
    base_seed=4321,  # Reproducible dataset definition
)

print("X shape:", dataset.X.shape)
print("Latent shape:", dataset.latent_trajectories.shape)
print("Relevant feature names:", dataset.relevant_feature_names[:5], "...")

## Step 2: plot the ice-cream

The left and middle panels are the cartoon. The right panel is the actual latent data
produced by `mltsa`.

In [ ]:
theta = np.linspace(0, 4 * np.pi, 400)
radius = np.linspace(0.15, 1.2, 400)
spiral_x = radius * np.cos(theta)
spiral_y = radius * np.sin(theta)

grid = np.linspace(-2.0, 2.0, 160)
xx, yy = np.meshgrid(grid, grid)
cartoon_surface = 0.18 * (xx**2 + yy**2) - 0.4 * xx + 0.1 * yy**2

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cone_x = np.array([-0.7, 0.0, 0.7, -0.7])
cone_y = np.array([-1.5, -0.1, -1.5, -1.5])
axes[0].fill(cone_x, cone_y, color="#d4a373", alpha=0.9)
axes[0].plot(spiral_x * 0.55, spiral_y * 0.55 + 0.7, color="#e76f51", lw=4)
axes[0].set_title("Ice-cream cartoon")
axes[0].set_aspect("equal")
axes[0].axis("off")

contour = axes[1].contourf(xx, yy, cartoon_surface, levels=18, cmap="cividis")
axes[1].plot(spiral_x, spiral_y, color="white", lw=2)
axes[1].set_title("Cartoon 2D landscape + spiral")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")
fig.colorbar(contour, ax=axes[1], shrink=0.8)

for index in range(6):
    color = "#bc4749" if dataset.y[index] else "#4c956c"
    axes[2].plot(
        dataset.latent_trajectories[index, :, 0],
        dataset.latent_trajectories[index, :, 1],
        color=color,
        alpha=0.75,
    )
axes[2].set_title("Latent trajectories generated by mltsa")
axes[2].set_xlabel("latent x")
axes[2].set_ylabel("latent y")

plt.tight_layout()
plt.show()

## Step 3: analysis of the ice-cream

Now we inspect both the observed features and the built-in ground-truth relevance.

In [ ]:
relevant_feature = dataset.relevant_features[0]
less_relevant_feature = next(
    index for index in range(dataset.n_features) if index not in dataset.relevant_features
)
time = np.arange(dataset.n_steps)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharex=True)
for index in range(4):
    axes[0].plot(time, dataset.X[index, :, relevant_feature], alpha=0.75)
    axes[1].plot(time, dataset.X[index, :, less_relevant_feature], alpha=0.75)

axes[0].set_title(f"More informative: {dataset.feature_names[relevant_feature]}")
axes[1].set_title(f"Mostly nuisance: {dataset.feature_names[less_relevant_feature]}")
for axis in axes:
    axis.set_xlabel("time step")
    axis.set_ylabel("feature value")
plt.tight_layout()
plt.show()

In [ ]:
from mltsa.synthetic import (
    plot_ground_truth_relevance,
    plot_relevance_over_time,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_ground_truth_relevance(dataset, ax=axes[0])
plot_relevance_over_time(dataset, ax=axes[1])
plt.tight_layout()
plt.show()

## Step 4: save the ice-cream for later

In [ ]:
from mltsa.synthetic import load_dataset

dataset_path = DATA_DIR / "synthetic_2d_icecream.h5"
dataset.save(dataset_path, overwrite=True)
print("Saved to:", dataset_path)

## Step 5: reconstruct the ice-cream

Because the seeds and system definition are stored, we can rebuild the same dataset
exactly.

In [ ]:
from mltsa.synthetic import load_dataset

reloaded = load_dataset(dataset_path)
rebuilt = reloaded.rebuild_exact()

print("Reloaded shape:", reloaded.X.shape)
print("Exact reconstruction:", np.allclose(reloaded.X, rebuilt.X))
print("Time relevance available:", reloaded.time_relevance is not None)